# SynthRare — Aviation TimeGAN/CTGAN Training (Google Colab)

Trains on aviation telemetry time-series (altitude, speed, heading) and uploads to HuggingFace Hub.

In [ ]:
# Cell 1 — Install dependencies + mount Drive
!pip install sdv ctgan huggingface_hub pandas scikit-learn scipy
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Load seed data from Drive
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/synthrare/seed_data/aviation_seed.csv', parse_dates=['timestamp'])
# Keep only numeric columns for CTGAN training
num_cols = df.select_dtypes(include='number').columns.tolist()
df_num = df[num_cols]
print(f'Loaded {len(df_num)} rows, numeric columns: {num_cols}')
df_num.head()

In [ ]:
# Cell 3 — Train CTGAN on numeric telemetry features
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_num)

synthesizer = CTGANSynthesizer(metadata, epochs=200, verbose=True)
synthesizer.fit(df_num)
print('Training complete')

In [ ]:
# Cell 4 — Validate: KS test + domain constraints
from scipy import stats
import numpy as np

synthetic = synthesizer.sample(num_rows=1000)

ks_scores = []
for col in num_cols:
    ks, _ = stats.ks_2samp(df_num[col].dropna(), synthetic[col].dropna())
    ks_scores.append(ks)
    print(f'  {col}: KS={ks:.4f}')

print(f'Mean KS: {np.mean(ks_scores):.4f}')

# Domain constraint checks
if 'altitude_ft' in synthetic.columns:
    assert (synthetic['altitude_ft'] >= 0).all(), 'Altitude must be non-negative'
if 'heading_deg' in synthetic.columns:
    assert (synthetic['heading_deg'].between(0, 360)).all(), 'Heading must be 0-360'
print('Domain constraints passed')

In [ ]:
# Cell 5 — Save and upload to HuggingFace Hub
from huggingface_hub import HfApi

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
REPO_ID = 'synthrare/aviation-timegan'

synthesizer.save('/content/model.pkl')
HfApi().upload_file(
    path_or_fileobj='/content/model.pkl',
    path_in_repo='model.pkl',
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'Model uploaded to {REPO_ID}')